## Create a compute instance with custom conda environment

Note:
If you see the following cell has error:
```
AzureCliCredential: Please run 'az login' to set up an account and login to the tenant
```

relogin from powershell
```powershell
az logout
az account clear
az login --tenant 00000000-0000-0000-0000-000000000000
```

Reference:
* https://learn.microsoft.com/en-us/azure/machine-learning/how-to-manage-compute-instance?view=azureml-api-2&tabs=python
* create aml compute instance basic https://learn.microsoft.com/en-us/azure/machine-learning/how-to-create-compute-instance?view=azureml-api-2&tabs=python

In [1]:
import azure.ai.ml as aml
print(f"Azure ML SDK version: {aml.__version__}")

Azure ML SDK version: 1.32.0


In [ ]:
# get a handle to the workspace
from azure.ai.ml import MLClient
from dotenv import load_dotenv
from azure.ai.ml.entities import ComputeInstance, AmlCompute
from azure.ai.ml.entities import ComputeInstanceSshSettings
from azure.core.exceptions import ResourceExistsError

import datetime
import os
config_file_name = "swedencentral.env"
config_file_path = os.path.join(".", "config", config_file_name)
load_dotenv(dotenv_path=config_file_path, override=True)

from utils.amlauth import AuthHelper
settings = AuthHelper.load_settings()
credential = AuthHelper.test_credential()

ml_client = MLClient(
    credential, settings.subscription_id, settings.resource_group, settings.workspace
)

Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


In [3]:
# get the location of the current workspace
workspace_location = ml_client.workspaces.get(settings.workspace).location
print(f"Workspace location: {workspace_location}")

Workspace location: swedencentral


In [5]:
# load the public SSH key content from the specified file path
ssh_public_key_path = os.path.expanduser(os.path.join("~", ".ssh", settings.ssh_pub_key_name))
# print(f"Loading SSH public key from: {ssh_public_key_path}")
with open(ssh_public_key_path, "r") as key_file:
    ssh_public_key_content = key_file.read().strip()
print(f"SSH public key content loaded successfully.")

SSH public key content loaded successfully.


In [ ]:
surfix = datetime.datetime.now().strftime("%Y%m%d%H%M")
ci_name = f"nb-t4{surfix}"  # Must be unique in the workspace
print(f"length of ci_name: {len(ci_name)}")
print(f"ci_name: {ci_name}")
ci_size = "Standard_NC4as_T4_v3"  # Optional, default is "Standard_DS3_v2"

ci = ComputeInstance(
    name=ci_name,
    size=ci_size,
    # location=workspace_location,
    idle_time_before_shutdown="PT15M",
    idle_time_before_shutdown_minutes=15,
    enable_node_public_ip=True,
    enable_os_patching=False,
    enable_root_access=True,
    enable_sso=True,
    setup_scripts=[],
    release_quota_on_stop=False,
    ssh_public_access_enabled=True,
    ssh_settings=ComputeInstanceSshSettings(
        # admin_username="azureuser",
        ssh_key_value=ssh_public_key_content,
        # ssh_port="50000",
    ),
    tags={
        "app": "finetuning",
        "domain": "ml"
    },
)

length of ci_name: 17
ci_name: nb-t4202603310016


In [7]:
start_time = datetime.datetime.now()
try:
    result = ml_client.compute.begin_create_or_update(compute=ci).result()
    print(f"✅ Compute instance '{result.name}' created successfully with size {result.size}")
except ResourceExistsError:
    print(f"⚠️ Compute instance '{ci.name}' already exists.")
except Exception as e:
    print(f"❌ Error creating compute instance: {e}")
end_time = datetime.datetime.now()
elapsed_time = end_time - start_time
print(f"Elapsed time for compute instance creation: {elapsed_time}")

✅ Compute instance 'nb-t4202603310016' created successfully with size Standard_NC4as_T4_v3
Elapsed time for compute instance creation: 0:04:11.034991


In [11]:
iterator = ml_client.compute.list()
last_ci_name = ""
for item in iterator:
    # type amlcompute is compute cluster
    # type computeinstance is compute instance
    if item.type in ["computeinstance"]:
        print(item.name, item.location, item.type)
        last_ci_name = item.name

notebook-t4-sw swedencentral computeinstance
nb-t4202603310016 swedencentral computeinstance


In [10]:
import json
ci_basic_name = last_ci_name
ci_basic_state = ml_client.compute.get(ci_basic_name)

print(f"compute instance '{ci_basic_state.name}' details:")
print(json.dumps(ci_basic_state.os_image_metadata.__dict__, indent=2, default=str))

compute instance 'nb-t4202603310016' details:
{
  "_is_latest_os_image_version": true,
  "_current_image_version": "26.01.05",
  "_latest_image_version": "26.01.05"
}


In [12]:
# stop
# from azure.core.exceptions import OperationFailed
try:
    ml_client.compute.begin_stop(ci_basic_name).wait()
    print(f"✅ Compute instance '{ci_basic_name}' stopped successfully.")
except Exception as e:
    print(f"❌ Failed to stop compute instance '{ci_basic_name}': {e}")

✅ Compute instance 'nb-t4202603310016' stopped successfully.


In [13]:
try:
    ml_client.compute.begin_delete(ci_basic_name).wait()
    print(f"✅ Compute instance '{ci_basic_name}' deleted successfully.")
except Exception as e:
    print(f"❌ Failed to delete compute instance '{ci_basic_name}': {e}")

✅ Compute instance 'nb-t4202603310016' deleted successfully.
